# Full FastViT feature substitution test inside WHAM on 3DPW

This is the downstream A/B test. For each identical 3DPW frame, it runs the official stored HMR2 feature and the distilled FastViT feature through the same frozen WHAM recurrent core. It compares both outputs with 3DPW ground-truth SMPL rotations and measures student-versus-teacher drift.

Required Kaggle inputs:

1. The saved phase-three polishing notebook version containing the exact epoch-36 `fastvit_hmr2_best.pth` selected on validation.
2. Your **registered, private 3DPW dataset** containing `imageFiles/`, plus the official public `3dpw_test_vit.pth` from WHAM's dataset folder. Do not use an unlicensed raw-data mirror. Set `THREEDPW_ROOT` below to its exact mount.

The WHAM checkpoint is fetched into temporary storage and verified by SHA-256. If the locked product gates pass, the notebook also exports and zips the phase-three Core ML package. Only those useful artifacts remain in `/kaggle/working`. This rotation diagnostic does not claim the paper's PA-MPJPE/MPJPE/PVE numbers because licensed SMPL model files are not used.


In [ ]:
# Configuration — change THREEDPW_ROOT only if Kaggle changed its mount.
from pathlib import Path
import hashlib

THREEDPW_ROOT = Path(
    '/kaggle/input/datasets/nguyntrunglong/3dpw-model'
)
SAVED_NOTEBOOKS_ROOT = Path('/kaggle/input/notebooks/nguyntrunglong')
SCRATCH_DIR = Path('/tmp/wham_feature_substitution')
OUTPUT_DIR = Path('/kaggle/working/wham_feature_substitution')
SEQUENCES = 0  # 0 means every matching test track.
FRAMES_PER_SEQUENCE = 0  # 0 means every available frame.
STUDENT_BATCH_SIZE = 64

EXPECTED_STUDENT_SHA256 = (
    'f15875f3fed12538312f59956b6c93e9cca2ab41a9e8d87edf593dd85f311ab1'
)
def checkpoint_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
checkpoint_candidates = sorted(
    SAVED_NOTEBOOKS_ROOT.rglob('fastvit_hmr2_best.pth')
)
phase3_matches = [
    path for path in checkpoint_candidates
    if checkpoint_sha256(path) == EXPECTED_STUDENT_SHA256
]
if len(phase3_matches) != 1:
    raise FileNotFoundError(
        'Attach the saved epoch-36 phase-three notebook output. '
        f'Expected SHA-256 {EXPECTED_STUDENT_SHA256}; found candidates '
        f'{[(str(path), checkpoint_sha256(path)) for path in checkpoint_candidates]}.'
    )
STUDENT_CHECKPOINT = phase3_matches[0]
if not THREEDPW_ROOT.is_dir():
    raise FileNotFoundError(
        f'Change THREEDPW_ROOT to your registered 3DPW mount: {THREEDPW_ROOT}'
    )
parsed_candidates = (
    THREEDPW_ROOT / '3dpw_test_vit.pth',
    THREEDPW_ROOT / 'parsed_data/3dpw_test_vit.pth',
    THREEDPW_ROOT.parent / '3dpw_test_vit.pth',
)
PARSED_3DPW = next((path for path in parsed_candidates if path.is_file()), None)
if PARSED_3DPW is None:
    raise FileNotFoundError(
        'Add WHAM 3dpw_test_vit.pth to the private 3DPW Kaggle input, either '
        'beside the 3DPW directory or inside it.'
    )
print(f'Student checkpoint: {STUDENT_CHECKPOINT}')
print(f'3DPW input: {THREEDPW_ROOT}')
print(f'WHAM parsed 3DPW: {PARSED_3DPW}')
print(f'Results only: {OUTPUT_DIR}')


In [ ]:
# Kaggle supplies CUDA-enabled PyTorch. These do not replace torch.
%pip install -q timm==1.0.22 ultralytics==8.4.146 coremltools==9.0 'numpy<=2.3.5' einops==0.8.1 yacs==0.1.8 joblib==1.5.2 loguru==0.7.3 smplx==0.1.28 opencv-python-headless==4.12.0.88 scikit-image==0.25.2 tqdm==4.67.1


In [ ]:
# Materialize the reviewed evaluator and exporter embedded in this notebook.
import base64, gzip, hashlib

SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
embedded = {
    'evaluate_wham_feature_substitution.py': ('6063f5dd58480b0c0c7bd144fb56f10e4ee206a4e0e740ad8da8dac30a08dc11', 'H4sIACsHomoC/809XXPbRpLv/BUI9uFAB4QpynIcbri1XsfeZCvOuRzv5kHHQkHkUEIMAjAASqJ1vt9+/TEz6AFASkr2qi61ZYEzPT09PdM9/QXsn756uqurpxdp/lTl1165b66K/HTk+/7Lp3+bNKpuvGKzSVdpknk/vH0/85J87a3TukmzTK29N0nd/Cv94G1U0uwqVXtpXqdr5f36w8u30Wj04Uq1w8ukqmHI6ffvfvU2aaa8eleWWQqD1qpRqyYt8jrkSQw6/TNXuyrJRmmeNoAo/ZwgbEikXFbFLl9PmmrXXHm/vH33k1cVDfXXkfc+ufEqdQnUqsrMnG6TS5gyqdSIFgEAn3YpdjeFtyq25a5R/WUVudfAWtRtsmq8Otkqb1PBvzWtMcVlNyrHWZMs23vqOsl2CTCPBlVqtasq6CauwByV8m5S4POuYYqTulYNkPtjM6pUWVRNbRcxqctkpYDhyWVeAL0rYEleNIS3TEpV/UftvX33j3evn77712tvq5IayN3CXEAZ7OFotKmKrRfHmx2uI45h+TgBsC43bBqNTFt1SVtkfq/qa/N4ldRXWXphfvIfaIh2wELT+ltd5Oa53l2UVbFSdW1b9vaxSbeKCVsVcIp456PkYmWoewVcTC4yDVQmDU5uOt/BT+5o9mWaX5r2l/neLuW34kKQm++25R647OWlIGFrn4tqdeX8iPI82uzyFW8ojnzDM7778Scz3Y94jjQdOMa057lu/LTemjZ8Ho1evn/1w48fXr/68M/3r72F52/gkF2nTVwns2cx7DOe7fhqW83i65k/wrMSv/rPt29//IDAs4uzZ5tvvvn2m9NvT1bfPnvxzfMXz15cfDs9U+sX35xdnJw9Wz1LZt+ewZaP1moDNMW07ACPopojd8be5C/AgihfJ1WV7OcjD/5LN15ag9A2Sb5SDBxqJnxQeV1UY4bD/yoFhyj3CCgCmU1WV8E4WpU7+JcnG48EHEyV1DQV4x1r0uqrZHb2PEYVEODezmlLibq6qXi6dXqJqmdhTl7Eg/QEKD10LKKiVHngVxf+GHcJhqtk2xK8KSpvdbXLP4J8eikogSBLthfrZK4hI/hnHbzwnngn09kz/Wccehe+L5bd0hPtyjWIdUA4nbXq/it1y0+BWSyI6Ao1Q6aZW8/FFoTep7m3yYqkodXT01yipZYABvTQAGvh+CvsI6Dnz0KQpnK/eJNktYI1fBpbfu+226RKPw9RQPOu01VzDhwJeT7vv1GdLZmQTZbgNjxoUmAnbBP0T04c5txZVvqgOstM1f4cpwgQeVQDZeOwBQEllvuaLQyBLcFYwpRnUwARTEG40DubOkAgDQNA3545syW3ncmSWzvXF83B5Dat4yS/zFQMcrVNmiq9DdrGuSswyFLZwIwkyBpYyV1ZCprlMroG7VdUcV5UW4EwhC3ZLiYnofdRqRKfP1QoP4jnKsk2sUWmH5540+iMuust6E7bAUq1Dsbed96Jmjzn/lUCN6+hQm3LZh9n6UcV8IBxC3T+P4RraYFBTQRidtM/9p56bovAYVEAfd7EwHFrVH/awTUcIIJnL6IpDfuE92aVg+K1865gawL9WNSSBDjmLdOACTTn2HDPnMEE+dHiPY+iKPTmJ0wmmgOwFdV+AOZkzjDNTREjs2fRFEhtoewCIhAxe+jhWlcVQFvM0S4HQKU+k2AAmYM9Mx69qoq6PSWfFfzk/SG0DHMbeqA6PnfmABNubYkgNLyKKSwEd2DyeaBnhj37bgccvCl2fB7ooBGT227PTA+Z7Ad6aH4eAsYhCGOzb4/gXgWnsGWoWBbtbkbUAO3qOl25HdTiKBiL9GveK5745yJX/O8SmB5YmecNmgjmdfdxaLyZw6L5mo5WC4pAtGyCMMqXlQWqDWPSxc/XAbc+SG/oJfIIfXaBpfNltMpg1qDVuk8YJqJf5/PJbBl6zw0dYnahw0zrgyjZpFUN+rNWqwIM74VFqYk6hencplMtPzQQBryJUM+hAQ+6mJFJSbWIJZxlt+6deDwUpZ1azKZ1dCX08xSj9gonlap3h+T6Kq3WrZrBvQucRXY1Ce2EVoZg/XwMuuCE0SigmeH9pSrWqk5X8VpdVkrVMRqIvAVgIfMSS/BA0lV/LwAn2OWqcVtHBw8LXGHpNSr4FqH3V40jaqokr8uiVsQvq3IK0Ow4JAjM+AgdDrR9A1jJCSyFljRDVhgZgZ04iaaou0EtghUIllXZ7tYE+kIEEOx2+Fcl6xmwQ+v1BBU702HNFqQzVlUF12NymaCJGrdaoMs0ONbDfIsPXdOHWCj4tjggNc60wugBXfcMRU6fLpofr+Ih66FHXQ/P6dhh2z2HqKXJrNzh47pKNw2f1gFW8fHtdRzTAweZY+Y4yJee/nDHW1oOIngYQxzJRF7ACYWbVXv2GFb4pdmhTg/A2XtbrHdgOWjfA5gWxxhsiGMgJ9sQH1DFtz5BvQO7EnSvhRsLRZVtogvQDhcFSRW6mqBcFFgN8RZIzgLHs3DcQD/E8wdiCoKwZqs6RPc1JuJVvZjase2Eqyvw5lWGVoMzN/pksYlguORZbxP8HxgGHPhFgSGSY6NLHnS9KvLr2Tow04BYz16gtq3gV4zW+wI26CJNauN7dBH8vSp25c9o4p48p9EDIK9/+mfQb365TkrURy+vL98VRQZUBCwZPcg3oLga8AX7PT8le1Xx7DN09dDPOx0AA5YnlYQJtTs4wHJiooksxRe7zQZOg69lmhyYUFpwASF64PC6WdvRsIt2sD2b4NXeJNWajmbI8awHyS3Jrglodc+KRmrPS0B4z+dwsdP/Tmfzyels2a7BXtFrg0seqsDgGXejB2LcEx7XrhtsKtmCjNRKDPyzNUCwwPKtdaVWH8sC3Mi4DSIYe9Hwg39pVb8Dz/O8L/+h8H9f5vuldnxb/K3PBkQEnXlDsMzKOCtWpMoW/qrcwe7dqPTyqqnjIs+Mc8xOIKBJMdYJrAG0La4I1hv4stsfm/iMM+irhSfjSCI4k6S18t7vcoyuvcZr0xXkjf/6tgQkwPc7ieGr6gv4/RhE9e7kTNCOUZO7znq/+B1xIJWGRltfsY4jDIQGAi7S+4i6ELkuuHnu6+0V3T6YlDXq80Y4wKQd0/wy5rgGehIiwOCwcO7wTnj9qixWV9Dd3QBul/EBIOZSDUByu4RMVitVAnsHgG1XHx7Dbrzig+MEiBwPrE3XdOoGRopOJyRyldTqND46tA8zgGGrmgR6k8PjLYSNpUhvhg5DUwTamQt7uyrl/uYqgRuwqLTVRz8xUG5k/tGqwN74LD71vo4omghGpqqaYEqnLrDzaM1NYV2MRRL1dYT9JsL7swLHs/roeX/yyn0GlMwxSYIx7AVDTMDrxMzIpClADV2rjPV5rsctDAbh8GzLbEE+qG3SptxiGp0IvyZW2wuwFs5OZm1jHmd47dWLUwmISnmB10nbWOV5TK63/9MvH9762j2Sgvt/ogjpODuS+1Ht5xxmdMK30BxyM2ojqS5oF/xlBOK9rYX9BRoTMyQwEJ20qqkxWgwCC+yMtFLls7hNa3A4LkNvlyujGhdmR3qaih6tOhJLgfk0Jg/obXE9RjVzYqhlNCAEQ3YFR1mjXtzphy+S3MVd+9xTzOa+1etphU2Klk7SxHTbBx3JIlkxuRh26fkSlRHs9tncnHgagJEWGbiIPojNUxabp2Dg4qxPjenxFHNIIIB7XkANy6HIlkwxRdjKBj6mDewxC3ytHKqijAkPWtBIutkaQpfWZL7j/uBv2ltVmeajG7XxXxW7bE1HSss6bxbM6OGMaQPn9o6vRnvT7SjG2lkDt/MqkIwA/xnbRWuqInULS2XYgP+MO6oTmiJn4+yOrvC4UlNcFUUT4D9iL/FBWzdJvkb1TpZgexoRHreLULxJMVLf+Rkeg+13YsrVfyzGoUEulIziUJLHLIeUhF3bvKsUbFcEbvc6BQ/OTfOAr9ikeUcF1eQarVAPVA5+VD3VABY8dmKMnYsS1iCwjSv+7BEI+MusuAh48fGT6LfyEu5QOqnOsDGeX1yUe4Y75rYldtQecOToz0XzBo2+jjoSp32TArWJSJrTRsC1VlHOYs9ZOEM4cCqD01t7rRra+MhOvtY9sRrvQmXFjXeHG6m1lk1VGTYwOEqVjo+1h9pc79dwmxZxuqZkXsj5ePjppLf4us/AzzrHQVpB2VlyTOQvLKaoqkuQZ1AqoEVOxufT5ai7NzrmzpTAEXVQjcRJGzwAXU3T34iN/1bfJcR3y13LddA1ErXROaCkY13ZAN4YaB2F/uHxMzW22hp41vIIEJwvrWQZtuLBtywWeUC4rZ4/EwtjQsklFFx7CoeB576jvJ9GNJ5Pz9ZfiBopqYwDuUY54o50EcFRUpYqXwcM2l7+KoPxU++7hefM433nZSoPWi4dw9lCnTtIlnKauiNy90pWKxU/F8xJz+XFF+L3nTmM7H25e+3Kv7wVaAFaiFoPmy/HNC93WuWAUUQGRicLfXFR3Lot4LkX2Y7D0V1xOmoD6OsF/DhKgiFmCiLMlt3k8akOP1O9UAs5m58eBF0VBejbXF9bmI574gV2TSY5svQmmgDEx1mcOaUpcSrZposJ1qhOrshSBbScC24ZAGpgHPZbT/RxyIpclypg5thBxgBiOyxbmHT986nFMZE1E3azzwlpC2ZptS1Lkz9rGSYuSGMsoXFbRqCT0UpAJgZBj7aQt8Ni1hlWmwO5HUAidsXGbFHa7MYAklMsqzCEdJFukxo9oM5Gwj5+502jU3nMbwdORkjjbZqraPRSwGNcFcFvveM+WAbDYEDEb/IonXxDea3+pLrsoQ1KlCq7Trvjz08w93mypGwdFTHwsmc2V1ULFHqJmpKJRgnr+is7XWm+URXpAjSkweHB9YIxUSWrxoTG2vCd9rKodT4QnAnFrdq7ArizpxV0fkQYnvNHuQcaLTo2FDemIpBQxzX7vnoo1Y2bsKBt0LoGHGvQb4Z+CSeuMpUlZU3+3VTnpMiyQ/8Q9SwWaAkrOIGzj1EAPMWCRVgZZIkXQZG1qlcLv1szKEzWTCXXSofzxU3J8QGyFvY1OKtgP6GtkDTN3sSxxT3V0JruWadZGXvksIG4us9p6d5DYk3nzIO55sXXYoVLN0ZOCvpB0J3LlexEKpfjci3yz6hcq9hVqyHT9fKC1YxRhQyIWuca4zP++7//DU1iq/N2cI5ejN2IvtVsoRdTOlQ6ukNThr1GupFmy34HXwbYPaN7ZTaFU9UHm509f0CjS7feZWOL9IbzvpMPySV3gk1yzd37YDw+MKm1GmVeW1MxbqMGWHaaxxegwj9iOKJVWzqaU3H0BB1nLFbbgPbdoRYWgRmZe9ARYm2MOcYfTRch+d5i4fmr3Trx3TOi8/XQEdX7fHVVFTmWCkjzjKX960GCQLdqgmVJCOoQw3VDKTPP1DgOZLGxNEmPhfOop71HZVe7vBvR1DGaeRuY5CPSqRAJ7W051G6IHuqjJOXH8mAXBhYPdrLT1e9cwRpAU16rrNvbqSp0dNVc3iJOxE0HhqA5uDXFLXy5j21mHhZxCqd7W1DiGP1LuAEFDt0BPAeGV4GdJjQsEKgokMCP15TCsKGyKvmNva14rRiRPSnuzGHLoVDwQxhfWBh+WSVubNE0gn/QxefmzYg43BxNJxlX+hmmM0/oZq8G2NCjvqUmbDe+X6Ih8ikIgFWUlpC2C7fCdNG2tF3IENOFz2h0n8yXMnuhQJWkzd5AwW+ZdcF1mi5edNsJC1dVYnqRDaKPWWE7+adbzWnrDnSqIaAik2NlsZgWnB9gEHqLbLNhIQJM3JbbMt5xhy8HwJBF6Gm0wBfFen8EGPjZKVPdJmkedIoVqKIfXQ5T3R+9rC53+IbAO+oJpNmwxRx6xcUGi96A79Um2WVN/YPKyjcGWBwenipK1us40UMCfzLh1z4mp+vyBvPaeC1xAMe8eCHN6GEUzVWl1AQQTOhg/U4s+tKZtHH234sJ1fcEQ9t/CMEfp8NEB2qDIKV0Nu/T4mR6dDC/wDI48nQ6fRAnyfqboPU3iOb5s6NYwF+eoEqZYAFPwrlGfDa46PIVy4mm96Iz9Ws9vAdwTqOT+5E2KoGtqiZUQ3WEwLNjBLZiPZm0aV2gM0tX+wm4UCrzpUfBKH3UYgrkkVSkzcjGPEwMuAKZXPg/7LZJPsGXGtCzwEv2Gnx0jB/SBGD2FPiiExnk+AYRylV9VWRr46wc5QUbOr/3rIINZs/rAKrjRzVJs0mBK8N7GYYlK04y0oLiBuY2cdDqEu1LjYf+ICbMD2r9zfRSZNHNfSAYj1jHqKxCt8d42K3UdgDYpjvaO5ANw3bMfUn+m3QieHWYh2Z/yXh16NB1VqFDzpzCNkHTZSc52ZrQmjl0kwQ23mxfgyOcc88H7w447Ud4uQUaibaBtfm63aYU6bXvekW0/Jh3VwSw/MsUd82v1DVfB/jjh9cvv8cCj9XNeuGyCI4F2EJ0pHQ6GrOuZWCTenL+rxaeeFXqd9XD6NfyCN2dQNYWxogJe3lWkQ9YDKTgaGl0gcVwqKgtojDitTIuBbs81gXjnwF7Prhc4e9gteg1SAPKN1z1GIf2KNVu7EUKKfvf63DKHeP68mdgx418/xG6Whq/+BhR2dVXWoo5aogagyNa+C4d5/y7MjKWYabQPMSmmIMYIgq3DgmSJ2v6Tb94U+CuX7nTnepc9ywHanf6sNwjYW39iSjj6Y+zncvjhTyHRgqg5WApTwz7FHMhNKAJ+nhEWc4S89l3X8ZUduPWlLZYuBqxPbLHioHk2l2EfTr6VUKanHYqt/YpBvz9mqIvzrHVsUlRCQHnFF8rjda7bVkH3aMx7p5ct6SmUzzkKplWCMNh/e0cSid8ZFC7MaVD6LU8UT0H6mGO3+Ebdm6mLc3X6jZEQY1NDgh1vcp36O40KmCJhCOQwvkTMTYLvaCyJYnAiawETubycNJzfCA5blZgoiRE8FgmPe0aH6yFMRkm3wu3CU/Mp9aMEF/NvrJvi68Hk9E9pbzRt7axkDElOJ3LClxT8GOIHvUTe/y2IkeOEBIuQhcrh4gNBhFjA/szpdeaOZCZpTm9yO0uftoZjq9UULn3Qk4sYnmUbB0uRzarOTfYKHdpyBgv2X7QP/FcWQqXI5tB1pZu3IaSjsWacUQ373D/CK7tfQigJoZjUw8l5H5oMERjs4EaslMALOSSlxXzWwS1SB+QoLfNd74mF1QnxoI9U83Kv7+MXDHHDTCbdlCQHXk/p3EiBW2zZWQ36GCwGfKxnA2NwRD+EDi2D4DbfP/QGNM5MK7Nkg4NxNSIGYPaprWTjWGjRc2RFTehSJLS68dlHOiySznQb1dlkjd2VcfhKTx2ANZRvqQ32OX2/iI10dDKBXDYdrr47BBXsQ1XMdHq64ytTPobnAjMYOqfCN2FeSFZwdErxnHzSBz9NIdXVOKci3mXQ2qrozzM2zo4+fHEZgdD2C/mGMhdSWo6I+RF7nY5Jmub7XKBTL6yv8SuAvl6YdYo8nqu0rV+gEjydNIfXVHsHVpnrYeqKVxKb2NQoxxuj/kV8uE6EiuEXDYiS0UEsqFF0AwOZWPxAvR0LEtUW5uDSgN6uAyZj0SnkwCn/PpZp1Sgx1cLLRgrFZbuHyiJsCPlJxFC5jHXk1AlwLSDi+q7xRvtjz4JFotD8dEDYCnkdw77BLGJNPASsysnQy85Di1qaOoTfrOc3vFr5cK8P/lIHnSW/2BJsNRYtWjew+yQZHz938sVxnI+pXj/QW4ITmDgge7SO9d21p4OmBj6ydVKPn7Z4rbThoIDzfin06PPrD/vM1x3DbDPzSp3lvCik5OXL3wMzK1TTfb5HnCdYxJ7cnhAm5qzy9Mv5olbEHhuvzfQQ6MtuIdnu41a52AY6pt+3rebGV9074JDCi30njzhUzFsbfw7kujSwj235u3y4Un1h/HJ3K2P4lPXKvj/wydj9j+AT72jorWte3KMQovEAqkcocdBPdxl6P3DtQBBI87NHwroQGiCKGKM0fWjb+bL1YQCe5/gh+GTyxvGR4kZg4ffdO8MEiRJtaq/e/Am0rGyOt2CKqjSZh/0bdOubLofh+h6loMFPHe9ch7fwINeshZ0H0rn6uZCWQ0AaVtTBv7Mp424rf8lpTb+KVik94JTzzzeOQJH0Ei+D6Bxdv4IGt7JNn13LxIQq8eS6KyYTo0zC7U8Ag2n9ofRcG5/GNmXAc3E8VAR+JQn63xywgXJnYD9oSCOOYnd9rEzqqdR9ahuuztKB3PsFAPC2Q3ldMlxZbIbyumSwdAy5uiEc47FHTHUmBf5hL55xdFGGFvjlx3B0/xYezeqUvbTjWtfT3PQPcParyF+u1mMg8OGGdsxeu0czOfxaOi6GKSmZWvnehimogX/48re/RjgH1f2Lr7HKXtn6L9L4TtIjdattsRerS65lvJT1QRvIuiIM/y20P1TjMfuJqDGaJEOKTiHxw74oK7V+UarWIW9QIMnztTON4Vid5j89ZTefZAjQ/zi3AuxYy5tjnIloEv9UoesrLqgwIKKu7cBKFc5+3cLjtIADfHQxSHrr8xaBlAOrdOOlHMcROKkHg9dMkh8yxCJ18DaS0SmyEwaDSu5siwgdkX8Vca2MpVeT3V4WAPGbRJfg6LjVZ4MfLEg3tA1jaZw3H5pFq0NDSDG0MTQRX9l/rQtJZl3/VSR++TClZjrXea89gPdPcd1eHcNlvt336I5dgoes8sW4b0mxX17TPssj8+qoPrHDiPxewq1wiPkdz9rTF9q1rXcX7ffXKbSBp1gw5cTOEjid9bA92DM9yBgxwi3vFe77rQ1RxFQK+kuSJaWsb5MmbOd1x94ndsyi/WnZw6AbBW+cR/DVa96aWkCeN/5VHJxk/PXXcVXk//c+2iy30f07uWEP6XcflC5XUDUeflueN/aiqv+5okaEugUv8IhuPbjCvzlWywqEN/NHcpYd7egX2hxBFcf+BA6UwMwUObQPVZtociRmWU5yTBX9e3YZ6ktlGjrY/VH5DqU4OWMJrm4q4enMvJJrL2uY/68eEyfF8c7ols67NzJ44EL4KGInNt6CNE2zXd13GYZO6xwHa6OgjysvVwt1xt/4FoMB+Y2l1tvZdRxYGcvy12MgUsTP+mvq2MzxTovxLR1sitDgtS1sIzo9QNbQ6O7FltvtA33HNAHQonyt4jtT+dyRzngeA2KAyCMth+xAoN/8FuCoafATm7i4qModpEjb8COVTGWzknXka2CkPLOebOYjbGu77+wIJZeksCXefxds5m8MCVkgsZYf8qD5VR26GgdWPQOBWlzBfsOl89t4Eer+tqX3+vuItYf78aq8FzdZCC2C3+IrKEve9Na0Z2ASaLv01XzKzUEDAfecaqyNdWQLDDD7zrQ06WIWzMmZh5+ucwJTsrOqrhxHXHjVGxLfvni7o9ZVo+4hO+5gA9Egvgk2HShvzw3ChSeCGY5oAl7kRzHzB/dH/mR/kU4+gMK6w8rq/tDP61lPjDqcaWHLszjygplVZwQZX3UhCy7VZ/vaYPn+B03K5NfOpWh70QZtse2FRaD9oTzi+8WUmEJNpbwcQk2f8kETCpDeTfy8ssejNPt69u0W5n4/vU/Xr/68Pp7+06y3mv9/5UhrDcyXc1tImw5Weo1GqX4rU4U9DimCH4c41swcayj+PxKzOh/AYgUi6TmZAAA'),
    'evaluate_mobile_pipeline_3dpw.py': ('9305461efdbe5c1fa0893b25536497643d636f5590695f3f8f56dc0820379ace', 'H4sIACsHomoC/7VbbXPbOJL+rl+B5X04ckPRb4k3pxlNnXfj7OQqTlwZ10xd6VQsmoQkjimSQ5CylJz/+3XjhQRIUE4ye6mUbYJAo9Ho1wfgv/3lpGHVyX2an9B8R8pDvSnyi4njONe7KGuimpJ6Q0lUltMsfaBkW9ynGSW//Xx1Q1hzz2hNipxUdJ2ymlY0IRdvbn8j6TZaUxZMJneblJG42JYZ3dK8ZoRKquHjJtqGKxrVTUVDpFSndVOnRR6Uh4DcbaJa9S0qkrIig0Fsgry8jVj9a3oHk5ZZFHO65DGtN6RYrdI4jTLyQA9lkeJ0UZ6QNE9raE0/R0j+B1gP8FTkdNIwCj3If398/5GUBaNkVRVAi8KYMmsYXziLtpTQfZkB5To7kJw2dQVTmERJJFhLP/6CogrIuxrZKypgoSpq3mfKSuCWJGm0zgtYbSy4SwpgIi9qEmdRusU5J2VU0urfGfnl5vY9ubn9r9vrk9tfrwnI4bGosmS6roomT0DWW1pXQCfA7ZpMgPstCcNVw0UawiYgAzBJLjlgk4lqq9ZlVDGqnmO2U3/+zopc/c0OTP1Zp1sqZiijepOl94r8LTyKF/WhTPO1ar/KD+10vxf3MEI95c22PIDESF621IsqllRu371XJN6hFknafyRb1Yx/i9Ymw704cFnKl7iZ7bzPKxuywV9i18lk8u7m6p/XH67vwpvrqw9kLhgLapqzonIXp8HL1698Ar9eXfJfp5dLL6go28COuRc+OYP/Xkfkl7s3Fhrn5/+Bg8/PX4pfr2w0JpOEroQdhXURytH8eSYEE/CfHpn+JGe4431mEwL/oqqKDjB3XgYR4w9irE8S2CY6h/ZVVkT1xbkXgAxzhvrvngNDODs5IeevXgWnnFRFQWo5ccUkKPeQ76DLycL8xBAaDtYFIFfC/mgikH1cFaXLyQ6X4vP2GMyZVuF+RjiDRuPBaGRpQnsNn6EB7B4Wfg47NOHC0WYQsmFxBB5szscDs5yAi2M9fb2cPyGbVVFtBdP4j3f1+WSe37aKWe5U9+Dq7dt3H6679934lgXfaDoNTs0GJQgQsOT0vN9lMMZCV4luhIy2BNDBCB31XKzlk3gEkw7+/u49LObqU9d3lWZZXGRFNXdPudKcSkJKc7dFQrMwobs0piG4mwbdtCueZ1JhxRPfJNy0/yWsluor9+CUpCsiegWotmQ+J07cJJFDaAb+2onLBhzf5D8FvTRfQQjKYUKc3fUkIxiwwgJCVbUTblDsBWo875jNuNeQOlQ3CXA667xCIOPNzzefzn8Rb/1Og0N0h2xGMoh/C/SFS7+jfh/V8SZs1dKYwf7SJiCpx3UDe7PQTd0nx56SNK4XIFEfXfFyKQS7D6viUbELTiBPuBUvwRwWS95jG7GHZzuhGasO+qywhx8gsuo9i3wFagebovpze9N6cEkxCh0TBq2g0xPeDmYEsorAjac59/mdCYGVrSkqXkZzV9sGz+/LXVPvhLJ47vBgryuD0/XIaLSj87cRqJY2LGXRPRgFhmiIhwGrE1pVAbTW9cFVWj9r+xdVuk5zoCGXq7kfbdGCoZrGIL1QZEvP9keB4DJRHtqiF0JIMymsF30RLGeGP+CZkpikKEF8SMPDMMiKpoqp2VlfEHpM3iWArdrRqnadT//8u+ONDmABpEKQTLmqAQZC2PCGI3qSUOMG/XTqGDJhda57+RIUAX7A5o86ruGU3qRt4nKDbGrOk5wAcq9VGEOGBW7T9XTvCOkGamjnOIISMl4wM5PT3mpMd5xu1+zzHHk2vTSYyRxU//TM79FCDzA/5k09c8QmylZzq880+8EO3sNC+ureLXjM9ZoaIqwfXwTskMcbyKBxVzoyhnm/sMoYQ5PYg4mh7GqrfSl8VPzPadnqE1MvmNfX8aTe+GRD0/UGk4FWZ1BjzO1YSRLBfbGHRBwqA3RgmGujb9HfeSiC06F9dB5Oei/9JYapY0NCzFApapU+FVrYCrx/HcUb1xOZGvyGaAc/RfI11Oh7yuowhbpgD9QgoLiY+1XrbbR3B/NZbNBYhphxMGzRzbHUDEgKUqPwI0riFUpRLquryKSEh0IRoUnZPjD/mVYFcy/+ZslZh+y3YUsjANMw9+zrxvOIpsYig8ck1Hbsmqzd6zRvqCknJYgwieoIJD10cn2JBdhTl/y3KIZnnRz1zWBkMfPJ7HxJ/qoXDAPOFtyuIIME9wU5pLIv+bwcyvkrWAkNvRswdb40nRZU3IneTxv9E+rchdEdjCncH/aHY3IWBoe9vlfG+O+44J4Xnk+eka2VZl/eviXMjT+ByaKb4DINWLOFyIxC/NvQNLdpnm6brSZ4tuDDlgG8cqN9yuanNpvcjw4DrzQ2TBU+fle7wP65iokXiq4nahkoYAvMnCy7wgueOfZ3xTaCqPBBMTZVK4OV/5WcBVD/ngU9jkY8uCrOlK90la4tTpfAYvt0vlRsjtE4WGicGTQuxmloK7RqiCCMrzWOYN0at74+kf7qbGnxk0LZQUqv7Cp51i9Je8mW8IxFlUA8rnngg3UBxSH7nbOa6sa1GKqHxfP09B7Eh5IyGrMCighWt9JHIRmJQ2/1WRELrG9+3ElaxSIW2WrNSTv7tDV82eI/P/5gjG89xVECsvYf7WPxMMe9iykcM3gPSAER8NQgP4q77rqaBnTQ10vI4JWQPQ8kjPO7o3s6ORr+XVuI+BEjhI2ySczIBfoQnA5kdQnqUCl9LnKvT/p4DiEKX4FTshY6FIkQ5qOcMxDT2SmCh2KDJC6ny4g7WYxkkL0jmYVICzGn5n/5fImYTlMIaLTCPRGkeR7H3wlQmlfzOle9Sn1QqBulucEHsD2EPrTEvWPXGDYob4dEuhyBt7VygxHxAzgMXJkQwbKTAa+i5ZRy0W0nffHegDj/DUFH1l8+9M3De9DcByg553dVoyWDX1dbFk1dNuiJ5NpcPsW/vBTrb+Gz1VhvIJcPCgyBCcGzmR5NepBPaIGhNQsYc9sKcZ6YK9OgZxghNlf4HU+rgcf7tj4CnMt9UWSuNkqtsGv5YsjWWVXRljJnNoSczH6i+OfAEof1XHcgjp+wNvJkxjU2XHNbjg5GwqhtVOG+9un2SfHCm59ogZdRew609IK8N2QFnnGX1lrnnsp0/Z9MuFcei4X8GGEU5R2eU+AC6rQ+tDZLD/wQxOLafAWGSIJCVeTRFozvZLSN6irdo89Wr8NLcORyqu6w5Uwctvjk0tA7NSig+zICBy36oMO99AJe1a2bomFue0gjCcNsRf0ti5fzPcd4T7W/UUL95V56zwDm7aHZDvQsyiW4lYP6ow+uxL4/8qSDzfjxn2jKonuKqKeJO0vMm2ag1jSRGCfYxVLH0FFwOqlvAOFzWj8W1YOSd54HN0XSqEOQ70ThubGHWbpN62/B5nsrF2vtwfASje0wRNBcxKRd6OJKqcqcIcqykFZVUdmQdh1Ah3SD0T8a3MiZddKubw16lYXCmWEI1xo7z9V7YQHoNVKjaQHfuxFcv43ArWK0Sr4DcypCXuWjSIRWLRxodpYLBT21bptvVMon6HQF7Ee4fjVYdbNQiHZRmiHAj1UU1LJmyg++vp0CA2MPncX3ag6UUkff7GtEck25MBSYMbzPjtbZ714a9LohPw7wyRaDatFYjFmGsJTmhFpQc3uQtbJRv90dvxP9Yqax8IKcLS3LhowTo6/fxVn9DCaE+MwxqdGzOgNM5mbTO/fUXYKxVn84vjN5K5HR98riLbsqLoOEOr4ms7Godzjg6in9qzN7Eu+TPRTmHp7hbecdoNx35a9fdyxAxramdTuvlv+YyPi4kZgKvDgj/X1VdZOtujsO+HWMtxRFQL345mOYNlXuVlI1ubjgERdVjzsZHebyt7mf+/l+cTZbBk0ONR2lkFJxLMl+rILqO8cfXz9E6fq8TZ+/eigqVPhQzgeKdXwEbuHckorZ+qI9z22pS69zDIKFmm5Hs7muumeaO4Kkop96/D8eKBlx5atLGBFIDb3hzoC3h9E6SnNWhwhJ4mqznh4JrVM2YhY9vrQ9y4q7CK6KffGk1YfK+25plLdQlByDbfpxqR7orVDLl4FpOjKJg1QeM7ghyOMogtCjde7DXm350236sJNR+wz8+0J/b4GbHG0/EroGEoZwzAFPFmkbuQ0ohoUDuYxlb4yW+tjH6Zz3xvbPNy2j7ZVYn5ClOretoFeiLYUDBdtCxEJLGbXLEimj5BOYBhjKNb50V86HgkhBqXQ/4YnZF9SSJ3mo39pMF8u6CcyzEJm5huDlz19dmgkGbwpXENnbBFfm05kW0Cdf5yI090C3JXiuOIo3yj0MgIMvlpKYz4Y1teAlGNqF1k8uqOsuG/oDaMVgj8Dvxg8KJNCttV+bi4sUUlfDNI+zJknzddjeKqWVMzMU+hjM0FfiUUwBgT5Lf3LCDxCM6ciZFU9o61KuBAYy0b4SEMVB+boemXXZhBiLNZDhy4gvUCbTsjwOXejwRSe5MfCiHYHcq0VYez0NEI++I+5de4M44vKSsDvk5jdtK7AJdes2uJI3OG75G1e/9LCNaoSP4yxibD4Y8IauIrzv8DPNyreq86Rzg2KqIEqS7paIM53y5mR6kZSPjk94wsmLbTCYP5q0oomGWo6QqDcVpVMgMMVU4XupyBR7CiYbP/CM5nsp4ZZN8Yr1nyLw5/k4FFmxez2V3uHPUDm//LNUlEq2BBC6AIfKdUadtI6MlZHx2weiTU55yTTFkslKQQG7z2jFM1QuXx6lInK075UdmHQrPwspichUa34BTJDgv5AIU8i3ms64b4AdRN8kRAv0zTeq6uxUsddBlDZjb4X+hVJzLC/PL0decrpoQeD+nSy9P+EBj51ge1AeHF/zK9uUMbzbPyeIKsk7g/p9xHblMhHB9gByaR71vaUK7pLOTMMjuCh5qHCdGzlNSw2HsxlxoP6E3XCC30ECriTiSaHL0m+LeMqchKG4+x+GrsOa+7IqwB6YA7k6SjAUO9ttzsJZp7jRTkV3wk3iw8/XV28gRSXxYzI3ZQUqQfc1VyeJroI80lLmH7BCnZu/6GkQfjET/uPjzc27u6O5mVlCOtf7kgNk4osbSfiLleyTD1vS5Il8Lfo+Ob1MWeRXbVInHl15qxoWoCVYWAupdB+CWnfl2utht0a+x49wFZzE60ouQh5AQtB/3oaAQJHtqCpvBAgBdMSnIkAkSty+5Xg6ctudJkLWBXkT106diwhSbXmc9qzFSaloGi8RgyFNC9Jgqki3NN9uwMPJtuj4pH3p56F0b9yHlGe3VfQYtiipcYZrgKVG4e1qm3XCgVWdihdUrMxScIShg5kfgk+4+0mqoJelXmUodo/qMZYYYu/EJ2EiNxZjoQ2/lgIWhBIRmIhfmT04rSGJrVIRrYdtKtiYX30RzEyGl3U4HiCBVJOcOJ5WQz3jtmqqMPC8DLI0519tqQPtdgTHd8UE2ikm7K4KUj0uF2oknn24ahZPnEerR9zMlgFZ2Mn6fXCuAjS/PMkjsPau/ijsj7PwOkeVMTiV9kmLI8JIzjNsx7eFFi2PF93PL4fdu2AzvBhfVrj0lfquENVdVJv4AeEXlK6Sl/ekq0wQBDDFKmvYpne4Lm7w+UpG/GwX1mw/wGohwUG5NwiPnUPqAcSSP38MHn8ek7YCkcLPHYOmDb91tJOtWhwHrpV2LVAqy/YWcidhvFEDAQ+RJl3KXm9XjenwC8IgabYlG0LFX6yXk55Fqex1rGB20X8xcllyiC+p8bbKdrlwEOv4naOuvP/SfrOLk0aAyrG+tjDzNGxCz57X8/4Vqx6e3JqAvpXCS7wWbPPLM3JLTZtefscytWUJN7ID+7bP0zqDf8k84tNZdHFdHwZhYxuFO1oxIAu7p52vwcuipAMkwcFbxIyiWjn9L5RrvDwnPtCA5LL9cphnWTJsCRfg9KEb6QRMsKd1Xb7l2og4vwsRNeCDEEYds1XH/LBYsA6MrdKK1aJKEx8sn7854Re1XhAE5GFRRAgXGtQ3yvz7Yf5h84vuooUo38054wgTCJyL04rydZNFFdlRyOVgzA8Q9cmb218/9seJ749DyAlQ+M6V+kgc54DiPIMl7Gh3R6P79PkHnkfcXk3F183HvnHGHQkcKxYl0o12ei6W28MdJrBtnIsqShom0U2pDySK4wY2AhZWbg4sjaNsmt5u8LuPf0BiR27eE/zaPI8PeBMMlJM1qDeMgmJAe3bQ+dGs2YFqY0fzSKDpPV3UcnKFOIkn39avzRY1ANKKqNoyzAHaOEh4n6M6cFAjifMxd+X083NE9HtN/W3tcv2vWrdeG9h3RHkpPN2Qf2pvZcoC+WHDQpnuDKJE5/SmrZ8VNJ5aUCCQt+GAIVhesH3AzFk8MOGxCd2D/wmLBx2KwHvrG54fajFTOD+/jQkeFr//I/2jPtljlUKSg/WoqyjBNHlcIJQ8d5p6NX0t8+mY7fjJvgAhWaADmLImRtMzqKc1JBkNOMa96wRAQJLi3w8qeuITQgdRxZw+QrZM545jYYJ/X1iDI9t2+SBnHyMJEAveQOb6G29wRT/I+FKaJZgOsDm/3o95BxYmXo+CkMOGRolxUKy/xKFul7f0cpaxfMUCS4/qiKEXgyGgQeNqNT7me/XSoCXUidte5Wob7NlO4DS1kCPUTnujuLg9f/FaWBxquTDEfQxDfqAThgiSh6E81BGI+eT/APzdfQOIRAAA'),
    'export_fastvit_normalized.py': ('e1c005e54f515afca3aab56b358b8281cff578a700cb6ff2c3f2254bce4f270f', 'H4sIACsHomoC/9Va/2/bthL/3X8Fp2F40p6sOk5SbN48IOuXtUDSBE26h4cgEGiJtrXo20gqqdflf393JCVRsux4w355RVtL4t3xeHc8fu6kr796UQn+YpHkL1j+QMqNXBf58chxnDefy4JLIteMxImQSZqymLylQv6a3JDHRK5JIgWRnCZ5kq9IXvCMpskfVCZFTpJ8vOK0XAcgaDRa8iIjYbisZMVZGJIkU6JpnhdS0YvRqH7GVyXlgtX3ZUrlEkTX92JdgSpaYknlOk0WtbgruNUDclOiSub5Wb5pxEcFZ1kqiyIVhAoSyXogr7Jyg4/ysn4kk6yZVhY8MsKv3p/Xkt9ndMXMlEhQP8/z0Wh09fHy9adXN+GvZ+fvX5/dvL/8EN68+/jm+t3l+etrMidfRgT+OBn9HJaFYGHMwGKxMgdeOzNyFByd+i0VZ2CL5IFtkQPpJDiaWKSS0WjNeBjzZCmNtNMAKJ5AsSilQpBztqLRxvjzWlYxy6Wb58FFEVcp82ZKWMyW4DdwsAxDV7B06ZHxT+RDkTM9jn9EVTLuekFD57VDwBEsaHS/AA5YMlo0iDijkoVZEbPUbUiV6kvQ5gFkCDo9cXxScqbCi8XztzQVzEcvhUp9JuaThrc3YcmL32AyWMs58FLudtQIUMYSVIBQFD45mkxPvFGzWAi1R8pjxeKDO8G/XrtU0KfieTtLV7Kryb3GxNcQoAlNjY3fXXyc7rIz7JIziJ9Esgj1IixLpITtttjUey+sbbPO+DSAUHVxV1BJHhgXuOOmnt5r/49ui9aQChhsyTnZ7auuekLbNlwzGmtnX7PfK7AtPOyqB0OvivxhGrv1NOD16Xc+uWcc7kKR/MHmRz5ZJFRohT2/L+AXXlTlBzC4e/RScQ+QvDn/5G4/PotpiZv27GF1BVkHtHBPfHLsbVO+hd0tWT4g45xuGNezT78j35IT+Hc8QKaj3aIx4e3v2imcrSC6IE8squUSosGRkH6ZDDNGc3CkymnBH4wXwlWCDmQXMm64wYsNcxOb9VnB4lAW97Dm4e1mPB9mtNyKDLNRm+jQm+925hP993g6Gx9P77zBvWsHj2vNcnAiaBdQK7a1JM3Sn95i/FYztjYj/+48QSdAKkFl0oLGQKFzhxKJR99MnXjauTRNi8cwVTl9RhYQaaCY3n16PIpYKUNIWnEVyfABdNBnR59YZQpZlSm7bZKUT4TkPmSiSN6qKzhR7+60NeCYie7LIsnlDB9jtlB+R5Vd1NInYNgwLSI13dyJygqC45Elq7UUYZGnG7PplLhkSRKR5ELSPGJuK1zP7gFkiIljLAEWwYyEAw4ADluVxuzUzqpziyQAE7uOPey0zgItOoxfzbtJLmwCCJJx+DB1Zp29CMlPMPKxgmSUsTecF9xdOp9yyLiIDsD1LZ6yp5mRL/btV/zJsePn9yrhEF0rWLNo4EOTgzl9DKMCTMecbmJwIjAVQ87hYStweSa2hhXW4Aam9QcXsHWEClSFSazhp+aq1rdvevUcIuHLk9elDUs8O+e9FQeJENVCAKe61ZEAUd/N9WrsNqcZu4M4Ije8YriTCT7BEOnKHEiL7cYYULkd7OmtdxeGbLigIkHtHTgFYwaHHs5Wg2Q9rdPwFTxZJTnEkRagsslWiJohx2tWhEtvTWVHLeDpbaHd4Ny7wnJNBTsOdy5UpR6TQ4YDUcUFXYgireQgUnW3yLtaGUWGELFPlpBWYDTJl47nDQr6cU72Au/b3YD7bktg74zVG20PCj9sbbsl/HML3D3HYausU2xdSGgxWE0cvMzdIv5hZ24XO8+u8alzZzbOzjOSYAZBAkg3negPgKYC7OF5s60ZBw6BwTU6H9myEliuyoKwz6YwJlXe5ISHqb1PyQ0U5PUeJ86wTEt3VBQOnFitAPNFQD6BYuOxlj82CxpbLDuE4lmtsiklKk+M5ZozZulG5BoKEgFCxDKB3ICtA1hRmkSJBNYdYiEu5HhdRKrcJRiqIPQFINdv4P/mgVGTlAWI2wTbsroRNJiRYaIQJhrwcDidTF+OJ9+PjyamhFIgV0cw8O6u5Lw+dWDAWg1NLBBzO4Rc7hS6gqs55na7JmrZQh0XYbsq524bAjQnxUwdE/2zGs0AQ33L9Mia08MyjiV463DpscesTItNhmu0uCwgOhCiQNDe9OGHcRWGmlgXaSwG88/eLLFFjfCuZ4UAPMKlwLaW2wSKs52VoHpkcCKOduSWdqDk4DmAfecQDbCHO8ju6QcAIlHBccBG4l/amy7005WLjh2/gxl9K05GIzuZ2QXB6NCkZCUj2Luam6zSYkHTcYl1grXZFyyiFUhLJFnSBMEs7oveLndMdWWfDnX+QQ3HZg6VWyABcqY9jimLrnJwRBIJa7drq7Qbc7CL1SHasx8Hdl7f1o7WL+ygf22QEA0SPhwpiNT0fH7eSKbagh8aXD3Y7VHxR14VYJqL838JMgmCYHp6Sj7+8jORLBeYZgHmtfZoGq1wkQts/og9DR+/XsKMtLP/9SZQa2hztb8B0HV9p4ug1+TeToKT7059WO7J6Uv1M3l55wW4uWnJ3CNfNS3gr3doz2LUwy3xwJTT6fc413R6on9On53y7/QBdMkPhxb4MZiQsdYXreDBQ2PPeLghUfcFG3meKf0z8Lrb85xqkHOYsW6WB2d8VWHSvVIjxouaLKBxHFIz7jrjsSm9wUx1NaTi38euOZtjS2Eve1HJspJ/l9ve9CCDRropIMBjDMqkqq7Ch9gbyzl7wItVgw4IbwfXLC3nveA5Q+X6qKWBJ10EMgNwagOWuuTp5z8LevvIMvnmG1KDc9IZxO3+47wRaFAtUag26Iu9WQOmiQsAWZjsM8rvSXv0kvro/UEtBrJ9UYGsGGQKiVWk6j40DrTTqzaQ9gEYHnGTcYX6QWcA3B3ZCXb3gQTM252rWnJg4tDyl3VkzRWJ/cTvVdsDKM4w7Rq217aV2IKorCARMiA320d1uoFgKKMbJq/LpdjYZ5qVKWsaYZCtY4QCEx/yAqQ8k3DUNfzn+SRWW0dTq5roeKo1gEwfqcyix35LZKAeuUo1v57KmDxUY9ZZFhX5A+OmqwAL1LetF7T41q5JDhtbzG87kQaMavE3oOM27sK+ytxReS9U7M52LanS7NzoGqi7baKoSAsepnQD6WWulIV7fRvAkdhlaBP1XTug8xKoD9w3KvMrlbWGdZc4xE6041lsGRx8WZWFFnDV/VfUQl8FyeX10cuWJSqyUjU3AMQl+O4FSZub4O355dlNTd/zREAruS4wdTv/eXd2MQbJTo8CUC7HKlZEPClNm6Y1/NJBkDVWLeYmiN0uuvTM21nzCrb7ZtaxN3gDggdaQQM1R9sL6jSRWoqBNtlzPTST7rdg/oDcgVqg1ag1IGBSDvZb4guoMGMSEywNqhJ+rAju1U6Pa5rVeSrsdIVn3fQ2wKabwIPlEuDL7U3TX1dD7x/Q/OuWJB6g20c88Hfq1S90jFJ9HfRYg3cdb7fEZyrEg5a8V4ZPrBcCh690uPQ8SJ0h1r+pRduH12/RtAZbLcAeFRh+n8l7LXxb8ui5Ntwgr5pvtKdBZk3eeQMQMiwazRcFB82/i/0vq7D9fcTh8293kg+ffF/D9yAN9vV7D1fjmXbqgbo825T9C04x6XjPByyDKm2nca3aTjl7lXoaWQWbxgC6MBKBvsNCr0gfmNu8YDTP2WeoIwHMzuwXKGYsEWGccLfX1tXfHgU8kwDQXU3aKoatoS69EVblaZLfm/nNM0DTCDqze5xG3whTSCm9wuJ+CMkFgsJK0Kxmdq9p+tQfSgViA+VxBjUjvrV8TfljklvvJ3VfquvTaxAaq1LB9CMA80f3gOmCulWERUAiN6p2uE/KEj9NMe2fXmFycX6B4BTxEHZbkEP1dugDTVK6AGAMmCGj0eX1D4RXOcwKFBpqQLFTIBbpSSxypdoFjWDOJSqo3g+nad2mMsqi4ORqjR8HOAPNA11ua3N9TW6wElLM1QLKOV3Yg2BQYgmo2Fdl1W+VITJA3PRl9IdtoxoKqtaXQtgKlovAmMB2k9+AxipPwNNA/Eo/+AT3waurT+Hlh/P/mh7UagHyDGC+ndwFJeMZkGLlMPXJxAsWUJJA2aG+WTNxpSCfrhPyIsQNZEcvGBjwi9JTf8BjpHdlQMFcUSx58jKggnJON24H9eJSa8+6Xzq4f6Y/hgvwYzjNCcvwnrzbHva+s/An7nh1IMCU+mUQTAwP8QfqabfRemxU8zzP7nECGb5pAAsy1xBgyQihDyJb4T/h13Gne5ugS6eNfBXoCojVfU0XZBEla/6lEfvk1a0K0+nVu+iL9vhTb7ArfobakT/r3ig4RNI/saHfSJ8FL5d9GdcZoP76oxhyPlXAHrnADimiqZX6CqU2xSw4ViJGowSbg1gIhSGZQ1IIQ+wqhaFJC7rFNPoft9zNOgoqAAA='),
}
for name, (expected_sha256, payload) in embedded.items():
    contents = gzip.decompress(base64.b64decode(payload))
    actual_sha256 = hashlib.sha256(contents).hexdigest()
    if actual_sha256 != expected_sha256:
        raise RuntimeError(f'Embedded script checksum mismatch for {name}')
    (SCRATCH_DIR / name).write_bytes(contents)
SCRIPT_PATH = SCRATCH_DIR / 'evaluate_wham_feature_substitution.py'
MOBILE_EVALUATOR_PATH = SCRATCH_DIR / 'evaluate_mobile_pipeline_3dpw.py'
EXPORTER_PATH = SCRATCH_DIR / 'export_fastvit_normalized.py'
print({name: digest for name, (digest, _) in embedded.items()})


In [ ]:
# Fetch the release WHAM checkpoint into ephemeral storage.
import hashlib, subprocess, urllib.request

SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
WHAM_REPO = SCRATCH_DIR / 'WHAM'
if not (WHAM_REPO / 'lib/models/wham.py').is_file():
    subprocess.run([
        'git', 'clone', '--filter=blob:none', '--no-checkout',
        'https://github.com/yohanshin/WHAM.git', str(WHAM_REPO),
    ], check=True)
    subprocess.run(['git', 'checkout', '--detach', '2b54f7797391c94876848b905ed875b154c4a295'], cwd=WHAM_REPO, check=True)
WHAM_CHECKPOINT = SCRATCH_DIR / 'wham_vit_bedlam_w_3dpw.pth.tar'
WHAM_CHECKPOINT_URL = (
    'https://huggingface.co/camenduru/WHAM/resolve/main/'
    'wham_vit_bedlam_w_3dpw.pth.tar?download=true'
)
WHAM_CHECKPOINT_SHA256 = (
    '2ba0cb6a7dd597023a6b2ad6056e7a8b6b33144a35fabea570bfd00842cd4eaf'
)
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
if not WHAM_CHECKPOINT.is_file():
    print('Downloading the checksum-pinned WHAM release checkpoint mirror...')
    urllib.request.urlretrieve(WHAM_CHECKPOINT_URL, WHAM_CHECKPOINT)
checkpoint_sha256 = sha256_file(WHAM_CHECKPOINT)
if checkpoint_sha256 != WHAM_CHECKPOINT_SHA256:
    raise RuntimeError(
        f'WHAM checkpoint SHA-256 mismatch: {checkpoint_sha256}. '
        'Delete the temporary file and retry.'
    )
YOLO_WEIGHTS = {
    'yolov8n-pose.pt': (
        'https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov8n-pose.pt',
        'c6fa93dd1ee4a2c18c900a45c1d864a1c6f7aba75d84f91648a30b7fb641d212',
    ),
    'yolo26n-pose.pt': (
        'https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26n-pose.pt',
        'eb3bb8268828aeaf515cec23a4bfafd793944a86fe9af94ba7823609c14522a9',
    ),
}
for name, (url, expected_sha256) in YOLO_WEIGHTS.items():
    path = SCRATCH_DIR / name
    if not path.is_file():
        urllib.request.urlretrieve(url, path)
    actual_sha256 = sha256_file(path)
    if actual_sha256 != expected_sha256:
        raise RuntimeError(f'{name} SHA-256 mismatch: {actual_sha256}')
YOLOV8_WEIGHTS = SCRATCH_DIR / 'yolov8n-pose.pt'
YOLO26_WEIGHTS = SCRATCH_DIR / 'yolo26n-pose.pt'
assert PARSED_3DPW.stat().st_size > 300_000_000
assert WHAM_CHECKPOINT.stat().st_size == 190_975_610
print('Parsed 3DPW labels and verified WHAM checkpoint are ready')


In [ ]:
# Run the controlled downstream substitution test.
import subprocess, sys

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH = OUTPUT_DIR / 'wham_feature_substitution_3dpw.json'
CSV_PATH = OUTPUT_DIR / 'wham_feature_substitution_3dpw.csv'
REPORT_PATH.unlink(missing_ok=True)
CSV_PATH.unlink(missing_ok=True)
command = [
    sys.executable, str(SCRIPT_PATH),
    '--parsed-3dpw', str(PARSED_3DPW),
    '--three-dpw-root', str(THREEDPW_ROOT),
    '--student-checkpoint', str(STUDENT_CHECKPOINT),
    '--wham-repo', str(WHAM_REPO),
    '--wham-checkpoint', str(WHAM_CHECKPOINT),
    '--sequences', str(SEQUENCES),
    '--frames', str(FRAMES_PER_SEQUENCE),
    '--student-batch-size', str(STUDENT_BATCH_SIZE),
    '--max-pose-degradation-deg', '1.15',
    '--max-relative-pose-degradation', '0.10',
    '--max-teacher-drift-deg', '5.0',
    '--acceptance-policy-label',
    'post_hoc_product_policy_locked_before_untouched_test',
    '--output', str(REPORT_PATH),
    '--per-sequence-output', str(CSV_PATH),
]
print(' '.join(command))
evaluation_result = subprocess.run(command, check=False)
print(f'Evaluator exit code: {evaluation_result.returncode}')


In [ ]:
# Full app-like 3DPW A/B: YOLO8/26 -> crop -> FastViT -> neutral WHAM_I -> WHAM.
MOBILE_REPORT_PATH = OUTPUT_DIR / 'mobile_pipeline_3dpw.json'
MOBILE_CSV_PATH = OUTPUT_DIR / 'mobile_pipeline_3dpw.csv'
MOBILE_REPORT_PATH.unlink(missing_ok=True)
MOBILE_CSV_PATH.unlink(missing_ok=True)
mobile_command = [
    sys.executable, str(MOBILE_EVALUATOR_PATH),
    '--parsed-3dpw', str(PARSED_3DPW),
    '--three-dpw-root', str(THREEDPW_ROOT),
    '--student-checkpoint', str(STUDENT_CHECKPOINT),
    '--wham-repo', str(WHAM_REPO),
    '--wham-checkpoint', str(WHAM_CHECKPOINT),
    '--yolov8-weights', str(YOLOV8_WEIGHTS),
    '--yolo26-weights', str(YOLO26_WEIGHTS),
    '--sequences', str(SEQUENCES),
    '--frames', str(FRAMES_PER_SEQUENCE),
    '--pose-batch-size', '32',
    '--student-batch-size', str(STUDENT_BATCH_SIZE),
    '--output', str(MOBILE_REPORT_PATH),
    '--per-sequence-output', str(MOBILE_CSV_PATH),
]
print(' '.join(mobile_command))
mobile_result = subprocess.run(mobile_command, check=False)
print(f'Mobile-pipeline evaluator exit code: {mobile_result.returncode}')


In [ ]:
# Compact result, conditional Core ML export, and download links.
import json, shutil, subprocess, sys
from IPython.display import FileLink, display

if REPORT_PATH.is_file():
    report = json.loads(REPORT_PATH.read_text())
    compact = {
        'accepted_for_device_diagnostic': report['accepted_for_device_diagnostic'],
        'gates': report['gates'],
        'thresholds': report['thresholds'],
        'scope': report['scope'],
        'feature_cosine': report['feature']['cosine'],
        'teacher_pose_error_deg': report['teacher_wham_vs_ground_truth']['all_joints_deg'],
        'student_pose_error_deg': report['student_wham_vs_ground_truth']['all_joints_deg'],
        'student_minus_teacher': report['student_minus_teacher'],
    }
    print(json.dumps(compact, indent=2))
    display(FileLink(str(REPORT_PATH)))
    display(FileLink(str(CSV_PATH)))
    if MOBILE_REPORT_PATH.is_file():
        mobile_report = json.loads(MOBILE_REPORT_PATH.read_text())
        mobile_compact = {
            'scope': mobile_report['scope'],
            'yolov8n_pose': mobile_report['variants']['yolov8n_pose'],
            'yolo26n_pose': mobile_report['variants']['yolo26n_pose'],
            'yolo26_minus_yolov8_pose_error_deg': (
                mobile_report['yolo26_minus_yolov8_pose_error_deg']
            ),
        }
        print(json.dumps({'mobile_pipeline': mobile_compact}, indent=2))
        display(FileLink(str(MOBILE_REPORT_PATH)))
        display(FileLink(str(MOBILE_CSV_PATH)))
    else:
        print('No mobile-pipeline report was produced; inspect the previous cell.')
    if report['accepted_for_device_diagnostic']:
        COREML_PATH = OUTPUT_DIR / 'FastViTNormalized.mlpackage'
        subprocess.run([
            sys.executable, str(EXPORTER_PATH),
            '--weights', str(STUDENT_CHECKPOINT),
            '--accept-product-validation',
            '--output', str(COREML_PATH),
        ], check=True)
        archive_base = OUTPUT_DIR / 'FastViTNormalized_phase3_mlpackage'
        archive_path = Path(str(archive_base) + '.zip')
        archive_path.unlink(missing_ok=True)
        shutil.make_archive(
            str(archive_base), 'zip',
            root_dir=OUTPUT_DIR, base_dir=COREML_PATH.name,
        )
        print({
            'coreml_archive': str(archive_path),
            'archive_bytes': archive_path.stat().st_size,
            'runtime_parity': 'must be rerun on macOS before iPhone deployment',
        })
        display(FileLink(str(archive_path)))
    else:
        print('Product test rejected the checkpoint; Core ML export skipped.')
    shutil.rmtree(SCRATCH_DIR, ignore_errors=True)
else:
    print('No report was produced. Keep the scratch directory and inspect the prior cell.')


## Decision rule

This run uses every available 3DPW test track and frame. The product policy was adopted after validation but is locked before this untouched test: no more than 1.15° absolute pose-error degradation, no more than 10% relative degradation, and no more than 5° mean pose drift from the official HMR2-feature WHAM output. Passing does not establish the paper's full world-grounded accuracy; it only shows that replacing HMR2 features with FastViT does not materially damage this held-out recurrent pose diagnostic.
